Read satellite data


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

GIMMS_PHENOLOGY_DIR = "../../data/satellite_data/images/PKU-GIMMS/phenology"
GIMMS_PHENOLOGY_PATTERN = "GIMMS_Phenology_SnowFilter_Forest1114_{year}.tif"
YEARS_GIMMS = list(range(1982, 2023))

def load_gimms_snowfilter_sos_eos(df, years, phenology_dir=GIMMS_PHENOLOGY_DIR):
        coords = list(zip(df["longitude"].values, df["latitude"].values))
    n = len(coords)
    sample_year = next(
        y for y in years
        if os.path.exists(os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=y)))
    )
    with rasterio.open(os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=sample_year))) as src:
        transform = src.transform
        height, width = src.height, src.width

    rows = np.empty(n, dtype=np.int32)
    cols = np.empty(n, dtype=np.int32)
    for i, (lon, lat) in enumerate(coords):
        r, c = rasterio.transform.rowcol(transform, lon, lat)
        rows[i], cols[i] = r, c
    valid_rc = (rows >= 0) & (cols >= 0) & (rows < height) & (cols < width)

    for year in years:
        fp = os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=year))
        if not os.path.exists(fp):
            print(f"  missing phenology file: {fp}", flush=True)
            df[f"sos_{year}"] = np.nan
            df[f"eos_{year}"] = np.nan
            continue
        with rasterio.open(fp) as src:
            sos_band = src.read(1)
            eos_band = src.read(2)
        sos = np.full(n, np.nan, dtype=np.float32)
        eos = np.full(n, np.nan, dtype=np.float32)
        sos[valid_rc] = sos_band[rows[valid_rc], cols[valid_rc]]
        eos[valid_rc] = eos_band[rows[valid_rc], cols[valid_rc]]
        sos[~np.isfinite(sos)] = np.nan
        eos[~np.isfinite(eos)] = np.nan
        df[f"sos_{year}"] = sos
        df[f"eos_{year}"] = eos
    return df

def _load_annual_climate_for_coords(lons, lats, years):
        key = set(zip(np.round(lons, 5), np.round(lats, 5)))
    years = set(int(y) for y in years)
    coords = pd.DataFrame({"longitude": lons, "latitude": lats})
    coords["_k"] = list(zip(np.round(coords["longitude"], 5), np.round(coords["latitude"], 5)))

    def load_chunked(paths, prefix):
        hits = []
        for fp in paths:
            cols = pd.read_csv(fp, nrows=0).columns
            ycols = [c for c in cols if c.startswith(prefix) and int(c.split("_")[-1]) in years]
            if not ycols:
                continue
            usecols = ["longitude", "latitude"] + ycols
            for chunk in pd.read_csv(fp, usecols=usecols, chunksize=200_000):
                mask = [
                    (round(lo, 5), round(la, 5)) in key
                    for lo, la in zip(chunk["longitude"], chunk["latitude"])
                ]
                sub = chunk.loc[mask]
                if len(sub):
                    hits.append(sub)
        if not hits:
            return coords[["longitude", "latitude"]].copy()
        df = pd.concat(hits, ignore_index=True)
        df["_k"] = list(zip(np.round(df["longitude"], 5), np.round(df["latitude"], 5)))
        ycols = [c for c in df.columns if c.startswith(prefix)]
        df = df.groupby("_k", as_index=False)[ycols].first()
        return coords[["_k"]].merge(df, on="_k", how="left")

    t_paths = [
        "../../data/climate_data/tables/climate_data/temp/temp-1982-1999.csv",
        "../../data/climate_data/tables/climate_data/temp/temp-2000-2024.csv",
    ]
    p_paths = [
        "../../data/climate_data/tables/climate_data/prcp/prcp-1982-1999.csv",
        "../../data/climate_data/tables/climate_data/prcp/prcp-2000-2024.csv",
    ]
    print("  loading annual T from climate tables...", flush=True)
    tdf = load_chunked(t_paths, "annual_t_")
    print("  loading annual P from climate tables...", flush=True)
    pdf = load_chunked(p_paths, "annual_p_")
    out = coords[["longitude", "latitude"]].copy()
    for c in tdf.columns:
        if c.startswith("annual_t_"):
            out[c] = tdf[c].values
    for c in pdf.columns:
        if c.startswith("annual_p_"):
            out[c] = pdf[c].values
    return out

def read_satellite_data(veg_type, satellite="gimms"):
        veg_class = pd.read_csv("../../data/veg_class_data/tables/veg_class.csv")

    if satellite in ("gimms", "avhrr"):
        print("  loading climate from climate tables + GIMMS snow-filter EOS...", flush=True)
        forest = veg_class[veg_class["veg_class"].isin([12, 13, 14])].copy()
        if veg_type in (12, 13, 14):
            forest = forest[forest["veg_class"] == veg_type].copy()
        df_clim = _load_annual_climate_for_coords(
            forest["longitude"].values, forest["latitude"].values, YEARS_GIMMS
        )
        df = forest.merge(df_clim, on=["longitude", "latitude"], how="inner")
        df = load_gimms_snowfilter_sos_eos(df, YEARS_GIMMS)
        t_cols = [c for c in df.columns if c.startswith("annual_t_")]
        p_cols = [c for c in df.columns if c.startswith("annual_p_")]
        eos_cols = [c for c in df.columns if c.startswith("eos_")]
        sos_cols = [c for c in df.columns if c.startswith("sos_")]
        df[t_cols] = df[t_cols] - 273.5
        df["annual_t"] = df[t_cols].mean(axis=1)
        df["annual_p"] = df[p_cols].mean(axis=1)
        df["eos"] = df[eos_cols].mean(axis=1)
        df["sos"] = df[sos_cols].mean(axis=1)
        df = df[df["eos"].notna()].copy()
        print(f"  pixels with EOS: {len(df)}", flush=True)
        return df

    clim_fp = f"../../data/satellite_data/tables/phenology_climate/{satellite}.csv"
    df_satellite = pd.read_csv(clim_fp)
    df = pd.merge(df_satellite, veg_class, on=["latitude", "longitude"], how="inner")
    if veg_type in (12, 13, 14):
        df = df[df["veg_class"].isin([veg_type])]
    else:
        df = df[df["veg_class"].isin([12, 13, 14])]
    eos_cols = [col for col in df.columns if "eos" in col]
    t_cols = [col for col in df.columns if "annual_t" in col]
    p_cols = [col for col in df.columns if "annual_p" in col]
    sos_cols = [col for col in df.columns if "sos" in col]
    if satellite == "modis":
        years = [str(y) for y in range(2001, 2024)]
    else:
        years = [str(y) for y in range(2013, 2023)]
    mask_sos = (df[sos_cols] < 0).any(axis=1)
    mask_eos = (df[eos_cols] > 365).any(axis=1)
    df = df[~(mask_sos | mask_eos)].copy()
    cols = years
    df = df[[col for col in eos_cols + t_cols + p_cols + sos_cols if any(y in col for y in cols)] + ["latitude", "longitude", "veg_class"]].copy()
    t_cols_df = [col for col in df.columns if "annual_t" in col]
    df[t_cols_df] = df[t_cols_df] - 273.5
    df.columns = df.columns.str.replace(r"\D*(\d{4})$", lambda m: f"{m.group(0)[0:-4]}{m.group(1)}", regex=True)
    df["annual_t"] = df[[col for col in df.columns if "annual_t" in col]].mean(axis=1)
    df["annual_p"] = df[[col for col in df.columns if "annual_p" in col]].mean(axis=1)
    df["eos"] = df[[col for col in df.columns if col.startswith("eos_")]].mean(axis=1)
    df["sos"] = df[[col for col in df.columns if col.startswith("sos_")]].mean(axis=1)
    return df


Read GIMMS


In [ ]:
satellite = "gimms"
veg_type = 0
df = read_satellite_data(veg_type, satellite)
df = df[
    (df["annual_t"] >= -20) & (df["annual_t"] <= 20) &
    (df["annual_p"] >= 0) & (df["annual_p"] <= 4)
]
print(df[["eos", "annual_t", "annual_p", "veg_class"]].describe())


In [ ]:
import matplotlib.pyplot as plt
import rasterio as rs
from rasterio.features import rasterize
import geopandas as gpd
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
from shapely.geometry import box
import matplotlib.path as mpath
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

def classify_rows(gdf):
    conditions = [
        (gdf['annual_t'] < 7.25) & (gdf['annual_p'] < 1.0),   # class 0: Cold-dry
        (gdf['annual_t'] >= 7.25) & (gdf['annual_p'] < 1.0),  # class 1: Hot-dry
        (gdf['annual_p'] >= 1.0)                              # class 2: Wet
    ]
    classes = [0, 1, 2]
    gdf = gdf.copy()
    gdf['class'] = np.select(conditions, classes, default=-1)
    return gdf

def show_classification_raster(class_raster, transform):
    fig = plt.figure(figsize=[6, 6])
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.NorthPolarStereo())
    ax.set_extent([-180, 180, 30, 90], ccrs.PlateCarree())
    ax.coastlines()

    theta = np.linspace(0, 2*np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform=ax.transAxes)

    cmap = ListedColormap(['#4d91c4', '#d47264', 'gray'])  
    labels = ['Cold-dry', 'Hot-dry', 'Wet']

    left, bottom, right, top = rs.transform.array_bounds(
        class_raster.shape[0], class_raster.shape[1], transform
    )

    class_masked = np.ma.masked_where(class_raster == -1, class_raster)

    ax.imshow(
        class_masked,
        cmap=cmap,
        extent=[left, right, bottom, top],
        transform=ccrs.PlateCarree(),
        origin="upper",
        interpolation="nearest"
    )

    valid = class_raster[class_raster != -1]
    total = len(valid)
    # counts = np.bincount(valid, minlength=3)
    # percentages = counts / total * 100
    valid = valid[~np.isnan(valid)]          # keep only finite values
    valid = valid.astype(np.int64)           # convert to integer class indices
    counts = np.bincount(valid, minlength=3)
    percentages = counts / counts.sum() * 100

    ax_bar = fig.add_axes([0.12, 0.10, 0.35, 0.25])

    bars = ax_bar.bar(range(3), percentages, color=['#4d91c4', '#d47264', 'gray'])

    ax_bar.set_xticks(range(3))
    ax_bar.set_xticklabels(labels, rotation=45, ha="right", fontsize=10)
    ax_bar.set_ylabel("Percentage (%)", fontsize=10)
    ax_bar.set_ylim(0, max(percentages) * 1.2 if total > 0 else 1)
    ax_bar.tick_params(axis='y', labelsize=7)
    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)

    for bar, pct in zip(bars, percentages):
        ax_bar.text(
            bar.get_x() + bar.get_width()/2, 
            bar.get_height() + 1, 
            f"{pct:.1f}", 
            ha='center', va='bottom', fontsize=10
        )

    import matplotlib.ticker as mticker
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
    gl = ax.gridlines(linewidth=0.5, color='gray', alpha=0.7, linestyle='--')
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 60))
    gl.ylocator = mticker.FixedLocator([30, 50, 70])
    lat_formatter = LatitudeFormatter()
    lon_formatter = LongitudeFormatter()
    for lat in [30, 50, 70]:
        ax.text(180, lat + 5, lat_formatter(lat), transform=ccrs.PlateCarree(),
                ha='left', va='center', fontsize=12, color='black')
    for lon in np.arange(-180, 181, 60):
        if lon in [-180, -60]:  # skip -180 and 60°W
            continue
        label_lon, label_lat = lon, 25
        if lon in [-120, 120]:
            label_lat = 23
        if lon == 0:
            label_lon = 2
            label_lat = 29
        if lon == 180:
            label_lon = 178
        ax.text(label_lon, label_lat, lon_formatter(lon),
                transform=ccrs.PlateCarree(),
                ha='center', va='top', fontsize=12, color='black')

    fig.savefig("../../results/ed_figures/ed_fig2/regions.png", dpi=500, bbox_inches='tight')
    plt.show()

def rasterize_classification(gdf, transform, width, height, crs):
    shapes = ((geom, value) for geom, value in zip(gdf.geometry, gdf['class']))
    # class_raster = rasterize(
    #     shapes=shapes,
    #     out_shape=(height, width),
    #     transform=transform,
    #     fill=-1,
    #     dtype='int32'
    # )
    class_raster = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=np.nan,
        dtype='float32'
    )
    print(f"[DEBUG] Classified pixels (excluding -1): {np.count_nonzero(class_raster != -1)}")
            
    import cartopy.io.shapereader as shpreader
    from shapely.geometry import box
    
    import os
    land_candidates = [
        os.path.expanduser("~/.local/share/cartopy/shapefiles/natural_earth/physical/ne_110m_land.shp"),
        os.path.expanduser("~/Library/Caches/cartopy/shapefiles/natural_earth/physical/ne_110m_land.shp"),
    ]
    land_fp = next((p for p in land_candidates if os.path.exists(p)), None)
    if land_fp is None:
        raise FileNotFoundError("ne_110m_land.shp not found in local cartopy cache")
    land = gpd.read_file(land_fp).to_crs(crs)
    
    left, bottom, right, top = rs.transform.array_bounds(height, width, transform)
    raster_bounds = box(left, bottom, right, top)
    land = gpd.clip(land, gpd.GeoDataFrame(geometry=[raster_bounds], crs=crs))
    
    land_mask = rasterize(
        [(geom, 1) for geom in land.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="uint8"
    )
    
    class_raster = np.where(land_mask == 1, class_raster, np.nan)
    print("[DEBUG] Pixels after land mask:", np.count_nonzero(~np.isnan(class_raster)))
    
    return class_raster

def show_map(df):
    with rs.open('../../data/satellite_data/images/base-image/test.tif') as src:
        transform = src.transform
        width, height = src.width, src.height
        crs = src.crs

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs=crs
    )

    left, bottom, right, top = rs.transform.array_bounds(height, width, transform)
    raster_bounds = box(left, bottom, right, top)
    gdf = gdf[gdf.geometry.within(raster_bounds)]

    print(f"[DEBUG] Points inside raster bounds: {len(gdf)}")

    gdf = classify_rows(gdf)

    class_raster = rasterize_classification(gdf, transform, width, height, crs)

    show_classification_raster(class_raster, transform)


Plot climate regions


In [ ]:
show_map(df)

Read vegetation data


In [ ]:
veg_class = pd.read_csv("../../data/veg_class_data/tables/veg_class.csv")
veg_class = veg_class[veg_class["veg_class"].isin([12, 13, 14])].copy()
print(veg_class["veg_class"].value_counts().sort_index())


In [ ]:
import os
import matplotlib.pyplot as plt
import rasterio as rs
from rasterio.features import rasterize
import geopandas as gpd
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
from shapely.geometry import box
import matplotlib.path as mpath
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.ticker as mticker
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter

def rasterize_classification(gdf, transform, width, height, crs):
    shapes = ((geom, value) for geom, value in zip(gdf.geometry, gdf["veg_class"]))
    class_raster = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=np.nan,
        dtype="float32",
    )
    print(f"[DEBUG] Vegetation pixels (non-NaN): {np.count_nonzero(~np.isnan(class_raster))}")

    land_candidates = [
        os.path.expanduser("~/.local/share/cartopy/shapefiles/natural_earth/physical/ne_110m_land.shp"),
        os.path.expanduser("~/Library/Caches/cartopy/shapefiles/natural_earth/physical/ne_110m_land.shp"),
    ]
    land_fp = next((p for p in land_candidates if os.path.exists(p)), None)
    if land_fp is None:
        raise FileNotFoundError("ne_110m_land.shp not found in local cartopy cache")
    land = gpd.read_file(land_fp).to_crs(crs)

    left, bottom, right, top = rs.transform.array_bounds(height, width, transform)
    raster_bounds = box(left, bottom, right, top)
    land = gpd.clip(land, gpd.GeoDataFrame(geometry=[raster_bounds], crs=crs))

    land_mask = rasterize(
        [(geom, 1) for geom in land.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="uint8",
    )
    class_raster = np.where(land_mask == 1, class_raster, np.nan)
    print("[DEBUG] Pixels after land mask:", np.count_nonzero(~np.isnan(class_raster)))
    return class_raster

def show_classification_raster(class_raster, transform):
    fig = plt.figure(figsize=[6, 6])
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.NorthPolarStereo())
    ax.set_extent([-180, 180, 30, 90], ccrs.PlateCarree())
    ax.coastlines()

    theta = np.linspace(0, 2 * np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    ax.set_boundary(mpath.Path(verts * radius + center), transform=ax.transAxes)

    cmap = ListedColormap(["#54a708", "#78d203", "#009900"])
    labels = ["DNF", "DBF", "MF"]
    class_codes = [12, 13, 14]
    bounds = [11.5, 12.5, 13.5, 14.5]
    norm = BoundaryNorm(bounds, cmap.N)

    left, bottom, right, top = rs.transform.array_bounds(
        class_raster.shape[0], class_raster.shape[1], transform
    )
    class_masked = np.ma.masked_invalid(class_raster)
    ax.imshow(
        class_masked,
        cmap=cmap,
        norm=norm,
        extent=[left, right, bottom, top],
        transform=ccrs.PlateCarree(),
        origin="upper",
        interpolation="nearest",
    )

    valid = class_raster[~np.isnan(class_raster)]
    if len(valid) > 0:
        counts = np.array([(valid == v).sum() for v in class_codes])
        percentages = counts / counts.sum() * 100
    else:
        percentages = np.zeros(3)

    ax_bar = fig.add_axes([0.12, 0.10, 0.35, 0.25])
    bars = ax_bar.bar(range(3), percentages, color=cmap.colors)
    ax_bar.set_xticks(range(3))
    ax_bar.set_xticklabels(labels, rotation=45, ha="right", fontsize=10)
    ax_bar.set_ylabel("Percentage (%)", fontsize=10)
    ax_bar.set_ylim(0, max(percentages) * 1.2 if len(valid) > 0 else 1)
    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)
    ax_bar.tick_params(axis="y", labelsize=7)
    ax_bar.set_yticks([0, 20, 40, 60])
    for bar, pct in zip(bars, percentages):
        ax_bar.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{pct:.1f}",
            ha="center", va="bottom", fontsize=10,
        )

    gl = ax.gridlines(linewidth=0.5, color="gray", alpha=0.7, linestyle="--")
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 60))
    gl.ylocator = mticker.FixedLocator([30, 50, 70])
    lat_formatter = LatitudeFormatter()
    lon_formatter = LongitudeFormatter()
    for lat in [30, 50, 70]:
        ax.text(180, lat + 5, lat_formatter(lat), transform=ccrs.PlateCarree(),
                ha="left", va="center", fontsize=12, color="black")
    for lon in np.arange(-180, 181, 60):
        if lon in [-180, -60]:
            continue
        label_lon, label_lat = lon, 25
        if lon in [-120, 120]:
            label_lat = 23
        if lon == 0:
            label_lon, label_lat = 2, 29
        if lon == 180:
            label_lon = 178
        ax.text(label_lon, label_lat, lon_formatter(lon), transform=ccrs.PlateCarree(),
                ha="center", va="top", fontsize=12, color="black")

    fig.savefig("../../results/ed_figures/ed_fig2/veg_classes.png", dpi=500, bbox_inches="tight")
    print("Saved ../../results/ed_figures/ed_fig2/veg_classes.png")
    plt.show()

def show_map(df):
    with rs.open("../../data/satellite_data/images/base-image/test.tif") as src:
        transform = src.transform
        width, height = src.width, src.height
        crs = src.crs

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs=crs,
    )
    left, bottom, right, top = rs.transform.array_bounds(height, width, transform)
    raster_bounds = box(left, bottom, right, top)
    gdf = gdf[gdf.geometry.within(raster_bounds)]
    print(f"[DEBUG] Points inside raster bounds: {len(gdf)}")
    class_raster = rasterize_classification(gdf, transform, width, height, crs)
    show_classification_raster(class_raster, transform)


Plot vegetation classes


In [ ]:
show_map(veg_class)

In [ ]:
import matplotlib.pyplot as plt
import rasterio as rs
from rasterio.features import rasterize
import geopandas as gpd
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import box
import matplotlib.path as mpath
from matplotlib.colors import LinearSegmentedColormap

def show_raster_with_title_and_colorbar(raster, transform, name):
    fig = plt.figure(figsize=[6, 6])
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.NorthPolarStereo())
    ax.set_extent([-180, 180, 30, 90], ccrs.PlateCarree())
    ax.coastlines()

    theta = np.linspace(0, 2 * np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform=ax.transAxes)

    cmap = LinearSegmentedColormap.from_list("custom_cmap",
        ["#0b3c68", "#165188", "#2066a8", "#4d91c4", "#8ec1da", 
         "#fbebe1", "#f6d6c2", "#d47264", "#c14d48", "#ae282c"]
    )

    left, bottom, right, top = rs.transform.array_bounds(raster.shape[0], raster.shape[1], transform)

    im = ax.imshow(
        raster,
        cmap=cmap,
        vmin=250,
        vmax=310,
        extent=[left, right, bottom, top],
        transform=ccrs.PlateCarree(),
        origin='upper'
    )

    cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
    cbar.set_label("EOS (DOY)", fontsize=14)

    cbar.set_ticks([260, 280, 300])
    cbar.ax.tick_params(labelsize=12)
    plt.tight_layout()

    import matplotlib.ticker as mticker
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
    gl = ax.gridlines(linewidth=0.5,color='gray',alpha=0.7,linestyle='--')
    gl.xlocator = mticker.FixedLocator(np.arange(-180,181,60))
    gl.ylocator = mticker.FixedLocator([30,50,70])
    lat_formatter = LatitudeFormatter()
    lon_formatter = LongitudeFormatter()
    for lat in [30,50,70]:
        ax.text(180,lat+5,lat_formatter(lat),transform=ccrs.PlateCarree(),ha='left',va='center',fontsize=12,color='black')
    for lon in np.arange(-180,181,60):
        if lon == -180:
            continue
        label_lon, label_lat = lon, 25   # higher so not under colorbar
        if lon in [-120,120]:
            label_lat = 23
        if lon == 0:
            label_lon = 2
            label_lat = 29
        if lon == 180:
            label_lon = 178
        ax.text(label_lon,label_lat,lon_formatter(lon),
                transform=ccrs.PlateCarree(),
                ha='center',va='top',fontsize=12,color='black')

    plt.savefig("../../results/ed_figures/ed_fig2/gimms_eos_map.png", dpi=500, bbox_inches="tight")
    plt.show()

def rasterize_data(gdf, transform, width, height, name, projection, crs):
    shapes = ((geom, value) for geom, value in zip(gdf.geometry, gdf['eos']))
    rasterized_data = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=np.nan,
        dtype='float32'
    )

    print(f"[DEBUG] Non-NaN values in raster: {np.count_nonzero(~np.isnan(rasterized_data))}")

    
    import cartopy.io.shapereader as shpreader
    from shapely.geometry import box
    
    import os
    land_candidates = [
        os.path.expanduser("~/.local/share/cartopy/shapefiles/natural_earth/physical/ne_110m_land.shp"),
        os.path.expanduser("~/Library/Caches/cartopy/shapefiles/natural_earth/physical/ne_110m_land.shp"),
    ]
    land_fp = next((p for p in land_candidates if os.path.exists(p)), None)
    if land_fp is None:
        raise FileNotFoundError("ne_110m_land.shp not found in local cartopy cache")
    land = gpd.read_file(land_fp).to_crs(crs)
    
    left, bottom, right, top = rs.transform.array_bounds(height, width, transform)
    raster_bounds = box(left, bottom, right, top)
    land = gpd.clip(land, gpd.GeoDataFrame(geometry=[raster_bounds], crs=crs))
    
    land_mask = rasterize(
        [(geom, 1) for geom in land.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="uint8"
    )
    
    rasterized_data = np.where(land_mask == 1, rasterized_data, np.nan)
    print("[DEBUG] Pixels after land mask:", np.count_nonzero(~np.isnan(rasterized_data)))

    show_raster_with_title_and_colorbar(rasterized_data, transform, name)

def show_map(df):
    with rs.open('../../data/satellite_data/images/base-image/test.tif') as src:
        transform = src.transform
        width, height = src.width, src.height
        crs = src.crs

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs=crs
    )

    left, bottom, right, top = rs.transform.array_bounds(height, width, transform)
    raster_bounds = box(left, bottom, right, top)
    gdf = gdf[gdf.geometry.within(raster_bounds)]

    print(f"[DEBUG] Points inside raster bounds: {len(gdf)}")

    projection = ccrs.NorthPolarStereo(central_longitude=0)
    rasterize_data(gdf, transform, width, height, 'EOS', projection, crs)


Plot GIMMS EOS map


In [ ]:
show_map(df)

Read PhenoCam data


In [ ]:
import pandas as pd
df = pd.read_csv('../../data/phenocam_data/tables/phenocam.csv')
eos_cols = [c for c in df.columns if c.startswith("eos_")]
df[eos_cols] = df[eos_cols].where(df[eos_cols] >= 200)
df = df[df['veg_type'].isin(["EN", "DB", "DN"])]
df = df[df['latitude'] >= 30]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

def get_long_term_mean(df, threshold):
    eos_cols = [c for c in df.columns if c.startswith("eos_")]
    annual_t_cols = [c for c in df.columns if c.startswith("annual_t_")]
    annual_p_cols = [c for c in df.columns if c.startswith("annual_p_")]

    eos_years = [int(c.split("_")[1]) for c in eos_cols]
    t_map = {int(c.split("_")[2]): c for c in annual_t_cols}
    p_map = {int(c.split("_")[2]): c for c in annual_p_cols}

    longterm_t, longterm_p, longterm_eos = [], [], []
    for _, row in df.iterrows():
        eos_values, t_values, p_values = [], [], []
        for year, eos_col in zip(eos_years, eos_cols):
            eos_val = row[eos_col]
            if pd.notna(eos_val):
                eos_values.append(eos_val)
                if year in t_map:
                    t_values.append(row[t_map[year]])
                if year in p_map:
                    p_values.append(row[p_map[year]])
        longterm_eos.append(np.nanmean(eos_values) if eos_values else np.nan)
        longterm_t.append(np.nanmean(t_values) if t_values else np.nan)
        longterm_p.append(np.nanmean(p_values) if p_values else np.nan)

    df = df.copy()
    df["longterm_eos"] = longterm_eos
    df["longterm_annual_t"] = np.array(longterm_t) - 273.5  # convert K → °C
    df["longterm_annual_p"] = longterm_p
    df = df[df["longterm_eos"] > 200]
    return df


In [ ]:
threshold  = 0.8
long_term_mean_phenocam = get_long_term_mean(df, threshold)

Read FLUXNET data


In [ ]:
import pandas as pd
df = pd.read_csv('../../data/flux_data/tables/flux_50.csv')
eos_cols = [c for c in df.columns if c.startswith("eos_")]
df[eos_cols] = df[eos_cols].where(df[eos_cols] >= 200)
df = df[df['latitude'] >= 30]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

def get_long_term_mean(df, threshold):
    eos_cols = [c for c in df.columns if c.startswith("eos_")]
    annual_t_cols = [c for c in df.columns if c.startswith("annual_t_")]
    annual_p_cols = [c for c in df.columns if c.startswith("annual_p_")]

    eos_years = [int(c.split("_")[1]) for c in eos_cols]
    t_map = {int(c.split("_")[2]): c for c in annual_t_cols}
    p_map = {int(c.split("_")[2]): c for c in annual_p_cols}

    longterm_t, longterm_p, longterm_eos = [], [], []
    for _, row in df.iterrows():
        eos_values, t_values, p_values = [], [], []
        for year, eos_col in zip(eos_years, eos_cols):
            eos_val = row[eos_col]
            if pd.notna(eos_val):
                eos_values.append(eos_val)
                if year in t_map:
                    t_values.append(row[t_map[year]])
                if year in p_map:
                    p_values.append(row[p_map[year]])
        longterm_eos.append(np.nanmean(eos_values) if eos_values else np.nan)
        longterm_t.append(np.nanmean(t_values) if t_values else np.nan)
        longterm_p.append(np.nanmean(p_values) if p_values else np.nan)

    df = df.copy()
    df["longterm_eos"] = longterm_eos
    df["longterm_annual_t"] = np.array(longterm_t) - 273.5  # convert K → °C
    df["longterm_annual_p"] = longterm_p
    df = df[df["longterm_eos"] > 200]
    return df


In [ ]:
threshold  = 1.0
long_term_mean_flux = get_long_term_mean(df, threshold)


In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
import cartopy.crs as ccrs
import matplotlib.path as mpath
from matplotlib.colors import LinearSegmentedColormap
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
import matplotlib.ticker as mticker
import pandas as pd

def show_sites_map(df_phenocam, df_flux):
    fig = plt.figure(figsize=[6, 6])
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.NorthPolarStereo())
    ax.set_extent([-180, 180, 30, 90], ccrs.PlateCarree())
    ax.coastlines(linewidth=0.8, color="black")

    theta = np.linspace(0, 2 * np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform=ax.transAxes)

    eos_cmap = LinearSegmentedColormap.from_list(
        "eos_cmap",
        ["#0b3c68", "#165188", "#4d91c4", "#8ec1da",
         "#fbebe1", "#f6d6c2", "#d47264", "#ae282c"]
    )

    bins = [0, 500, 1000, 1500, 2000]
    labels = ["0–500", "500–1000", "1000–1500", "1500–2000"]

    def add_precip_class(df):
        df["precip_class"] = pd.cut(df["longterm_annual_p"] * 1000,
                                    bins=bins, labels=labels, include_lowest=True)
        return df

    df_phenocam = add_precip_class(df_phenocam)
    df_flux = add_precip_class(df_flux)

    size_map = {"0–500": 5, "500–1000": 15, "1000–1500": 25, "1500–2000": 35}

    for label in labels:
        subset = df_phenocam[df_phenocam["precip_class"] == label]
        if subset.empty:
            continue
        ax.scatter(
            subset["longitude"], subset["latitude"],
            c=subset["longterm_eos"],
            s=size_map[label],
            cmap=eos_cmap, vmin=245, vmax=305,
            edgecolors="black", linewidths=0.3,
            transform=ccrs.PlateCarree(), alpha=0.9,
            marker='o', label=None
        )

    for label in labels:
        subset = df_flux[df_flux["precip_class"] == label]
        if subset.empty:
            continue
        ax.scatter(
            subset["longitude"], subset["latitude"],
            c=subset["longterm_eos"],
            s=size_map[label],
            cmap=eos_cmap, vmin=245, vmax=305,
            edgecolors="black", linewidths=0.3,
            transform=ccrs.PlateCarree(), alpha=0.9,
            marker='D', label=None
        )

    import matplotlib.colors as mcolors
    spruce_eos = 306
    norm = mcolors.Normalize(vmin=245, vmax=305)  # same as scatter plots
    spruce_color = eos_cmap(norm(spruce_eos))
    ax.scatter(
        -93.4541, 47.5049,          # lon, lat
        marker='^',                  # triangle marker
        s=100,                       # size
        color=spruce_color,          # color from colormap
        edgecolor='black',
        linewidth=1,
        transform=ccrs.PlateCarree(),
        zorder=10,
        label="Spruce site"
    )
    

    handles_phenocam = [
        plt.scatter([], [], s=size_map[label],
                    marker='o',
                    facecolors='gray', edgecolors='black', alpha=0.8,
                    label=f"{label} mm/yr")      # ← explicitly empty label
        for label in labels
    ]

    handles_flux = [
        plt.scatter([], [], s=size_map[label],
                    marker='D',
                    facecolors='gray', edgecolors='black', alpha=0.8,
                    label=f"{label} mm/yr")
        for label in labels
    ]

    handles_all = handles_phenocam + handles_flux

    leg = ax.legend(
        handles=handles_all,
        title="",
        loc="lower center",
        bbox_to_anchor=(0.5, -0.25),
        ncol=2,
        fontsize=9,
        title_fontsize=10,
        frameon=True
    )

    frame = leg.get_frame()
    frame.set_facecolor("white")
    frame.set_edgecolor("none")

    gl = ax.gridlines(linewidth=0.5, color='gray', alpha=0.6, linestyle='--')
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 60))
    gl.ylocator = mticker.FixedLocator([30, 50, 70])

    lat_formatter = LatitudeFormatter()
    lon_formatter = LongitudeFormatter()
    for lat in [30, 50, 70]:
        ax.text(180, lat + 5, lat_formatter(lat), transform=ccrs.PlateCarree(),
                ha='left', va='center', fontsize=12, color='black')
    for lon in np.arange(-180, 181, 60):
        if lon == -180:
            continue
        label_lon, label_lat = lon, 25
        if lon in [-120, 120]:
            label_lat = 23
        if lon == 0:
            label_lon = 2
            label_lat = 29
        if lon == 180:
            label_lon = 178
        ax.text(label_lon, label_lat, lon_formatter(lon),
                transform=ccrs.PlateCarree(),
                ha='center', va='top', fontsize=12, color='black')

    plt.tight_layout()
    plt.savefig("../../results/ed_figures/ed_fig2/phenocam_flux_map.png", dpi=500, bbox_inches='tight')
    plt.show()


Plot site map


In [ ]:
show_sites_map(long_term_mean_phenocam, long_term_mean_flux)

Plot PEP725 sites


In [ ]:
## PEP725 sites
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib.path as mpath
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

df_pep_raw = pd.read_csv("../../data/pep725_data/tables/pep725_revised_06052026.csv")
df_pep_raw = df_pep_raw[df_pep_raw["species"] != "Fagus"].copy()

eos_cols = [c for c in df_pep_raw.columns if c.startswith("eos_") and c != "eos_code"]
temp_cols = [c for c in df_pep_raw.columns if c.startswith("annual_t_")]
precip_cols = [c for c in df_pep_raw.columns if c.startswith("annual_p_")]

df_eos = df_pep_raw.melt(
    id_vars=["PEP_ID", "species", "latitude", "longitude"],
    value_vars=eos_cols, var_name="Year", value_name="EOS",
)
df_eos["Year"] = df_eos["Year"].str.replace("eos_", "").astype(int)
df_temp = df_pep_raw.melt(
    id_vars=["PEP_ID", "species"], value_vars=temp_cols, var_name="Year", value_name="Annual_Temp",
)
df_temp["Year"] = df_temp["Year"].str.replace("annual_t_", "").astype(int)
df_precip = df_pep_raw.melt(
    id_vars=["PEP_ID", "species"], value_vars=precip_cols, var_name="Year", value_name="Annual_Precip",
)
df_precip["Year"] = df_precip["Year"].str.replace("annual_p_", "").astype(int)

merged = pd.merge(df_eos, df_temp, on=["PEP_ID", "species", "Year"])
merged = pd.merge(merged, df_precip, on=["PEP_ID", "species", "Year"])
merged = merged.dropna(subset=["EOS", "Annual_Temp", "Annual_Precip"])
if merged["Annual_Temp"].mean() > 200:
    merged["Annual_Temp"] = merged["Annual_Temp"] - 273.15

TEMP_MIN, TEMP_MAX = 5.0, 12.0
P_THRESHOLD = 1.0

clean = (
    merged.groupby(["PEP_ID", "species"])
    .agg({
        "EOS": "mean", "Annual_Temp": "mean", "Annual_Precip": "mean",
        "latitude": "first", "longitude": "first",
    })
    .reset_index()
    .rename(columns={"EOS": "mean_EOS", "Annual_Temp": "mean_Temp", "Annual_Precip": "mean_Precip"})
)
clean = clean[(clean["mean_Temp"] >= TEMP_MIN) & (clean["mean_Temp"] <= TEMP_MAX)].copy()

bin_edges = np.arange(TEMP_MIN, TEMP_MAX + 0.5, 0.5)
bin_labels = [f"{bin_edges[i]:.1f}" for i in range(len(bin_edges) - 1)]
clean["Temp_Bin"] = pd.cut(clean["mean_Temp"], bins=bin_edges, labels=bin_labels, include_lowest=True)
clean = clean.dropna(subset=["Temp_Bin"])
clean["Precip_Group"] = np.where(clean["mean_Precip"] <= P_THRESHOLD, "Dry regions", "Wet regions")

chunks = []
for _, g in clean.groupby(["Precip_Group", "Temp_Bin"], observed=False):
    chunks.append(g if len(g) >= 10 else g.iloc[0:0])
trimmed = pd.concat(chunks, ignore_index=True)

site_rows = []
for pep_id, g in trimmed.groupby("PEP_ID"):
    precip = "Dry regions" if (g["Precip_Group"] == "Dry regions").any() else "Wet regions"
    site_rows.append({
        "PEP_ID": pep_id,
        "latitude": g["latitude"].mean(),
        "longitude": g["longitude"].mean(),
        "Precip_Group": precip,
        "mean_EOS": g["mean_EOS"].mean(),
    })
pep_sites = pd.DataFrame(site_rows)
print(f"PEP725 sites used in study: {len(pep_sites)}")
print(pep_sites["Precip_Group"].value_counts().to_string())

def show_pep725_sites_map(df_sites, out_fp="../../results/ed_figures/ed_fig2/pep725_sites_map.png"):
        import geopandas as gpd

    os.makedirs(os.path.dirname(out_fp), exist_ok=True)
    fig = plt.figure(figsize=[7, 6])
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    ax.set_extent([-12, 35, 36, 72], crs=ccrs.PlateCarree())

    def _first_existing(paths):
        return next((p for p in paths if os.path.exists(p)), None)

    land_fp = _first_existing([
        "../../data/shapefiles/ne_10m_land.shp",
        os.path.expanduser("~/.local/share/cartopy/shapefiles/natural_earth/physical/ne_10m_land.shp"),
    ])
    coast_fp = _first_existing([
        "../../data/shapefiles/ne_10m_coastline.shp",
        os.path.expanduser("~/.local/share/cartopy/shapefiles/natural_earth/physical/ne_10m_coastline.shp"),
        "../../data/shapefiles/ne_50m_coastline.shp",
        os.path.expanduser("~/.local/share/cartopy/shapefiles/natural_earth/physical/ne_50m_coastline.shp"),
    ])
    borders_fp = _first_existing([
        "../../data/shapefiles/ne_10m_admin_0_boundary_lines_land.shp",
        os.path.expanduser("~/.local/share/cartopy/shapefiles/natural_earth/cultural/ne_10m_admin_0_boundary_lines_land.shp"),
    ])

    from shapely.geometry import box as shapely_box
    extent_box = shapely_box(-12, 36, 35, 72)

    if land_fp is not None:
        land = gpd.read_file(land_fp)
        land = land[land.intersects(extent_box)].clip(extent_box)
        ax.add_geometries(
            land.geometry, crs=ccrs.PlateCarree(),
            facecolor="#f2f2f2", edgecolor="none", zorder=1,
        )
    if borders_fp is not None:
        borders = gpd.read_file(borders_fp)
        borders = borders[borders.intersects(extent_box)]
        ax.add_geometries(
            borders.geometry, crs=ccrs.PlateCarree(),
            facecolor="none", edgecolor="#888888", linewidth=0.35, zorder=2,
        )
    if coast_fp is not None:
        coast = gpd.read_file(coast_fp)
        coast = coast[coast.intersects(extent_box)]
        ax.add_geometries(
            coast.geometry, crs=ccrs.PlateCarree(),
            facecolor="none", edgecolor="black", linewidth=0.55, zorder=3,
        )
        print(f"Basemap: land={os.path.basename(land_fp or '')}, "
              f"coast={os.path.basename(coast_fp)}, "
              f"borders={os.path.basename(borders_fp or '')}")
    else:
        print("Warning: coastline shapefile not found; plotting sites only.")

    style = {
        "Dry regions": dict(c="#e03c31", s=3, alpha=0.35, zorder=5),
        "Wet regions": dict(c="gray", s=3, alpha=0.55, zorder=6),
    }
    rng = np.random.default_rng(0)
    jitter_deg = 0.12  # ~13 km; separates stacked / co-located sites
    for grp, kw in style.items():
        sub = df_sites[df_sites["Precip_Group"] == grp]
        if sub.empty:
            continue
        lon = sub["longitude"].to_numpy(dtype=float) + rng.uniform(-jitter_deg, jitter_deg, len(sub))
        lat = sub["latitude"].to_numpy(dtype=float) + rng.uniform(-jitter_deg, jitter_deg, len(sub))
        ax.scatter(
            lon, lat,
            c=kw["c"], marker="o", s=kw["s"], alpha=kw["alpha"],
            edgecolors="none",
            transform=ccrs.PlateCarree(), zorder=kw["zorder"],
            label=grp,
        )

    gl = ax.gridlines(
        draw_labels=True, linewidth=0.4, color="gray", alpha=0.5, linestyle="--",
        x_inline=False, y_inline=False,
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 11}
    gl.ylabel_style = {"size": 11}
    gl.xlocator = mticker.FixedLocator(np.arange(-10, 40, 10))
    gl.ylocator = mticker.FixedLocator(np.arange(40, 75, 10))
    gl.xformatter = LongitudeFormatter()
    gl.yformatter = LatitudeFormatter()

    from matplotlib.lines import Line2D
    legend_handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor="#e03c31",
               markersize=12, label="Dry regions"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor="gray",
               markersize=12, label="Wet regions"),
    ]
    leg = ax.legend(handles=legend_handles, loc="upper left", fontsize=16,
                    frameon=True, handletextpad=0.6, borderpad=0.6)
    leg.get_frame().set_facecolor("white")
    leg.get_frame().set_edgecolor("none")
    leg.get_frame().set_alpha(0.92)

    plt.tight_layout()
    plt.savefig(out_fp, dpi=500, bbox_inches="tight")
    print(f"Saved {out_fp}")
    plt.show()

show_pep725_sites_map(pep_sites)
